<a href="https://colab.research.google.com/github/BernardoBremer/Inteligencia-Computacional-Cetys-/blob/main/2_1_evaluation_sample.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Metrics and Evaluation - California Housing Dataset

In [26]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

In [27]:
x, y = fetch_california_housing(return_X_y=True, as_frame=True)
x.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


In [28]:
x_train, x_temp, y_train, y_temp = train_test_split(x, y, test_size=0.2, random_state=42)
x_val, x_test, y_val, y_test = train_test_split(x_temp, y_temp, test_size=0.5, random_state=42)

print(f"Train: {x_train.shape}, Val: {x_val.shape}, Test: {x_test.shape}")

Train: (16512, 8), Val: (2064, 8), Test: (2064, 8)


In [29]:
non_geo_idx = [0, 1, 2, 3, 4, 5]
geo_idx = [6, 7]

x_train_t = torch.tensor(x_train.values, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)
x_val_t = torch.tensor(x_val.values, dtype=torch.float32)
y_val_t = torch.tensor(y_val.values, dtype=torch.float32).unsqueeze(1)
x_test_t = torch.tensor(x_test.values, dtype=torch.float32)
y_test_t = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)

In [30]:
train_non_geo = x_train_t[:, non_geo_idx]

Q1 = torch.quantile(train_non_geo, 0.25, dim=0)
Q3 = torch.quantile(train_non_geo, 0.75, dim=0)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

train_capped = torch.clamp(train_non_geo, min=lower_bound, max=upper_bound)
train_mean = train_capped.mean(dim=0)
train_std = train_capped.std(dim=0)

In [31]:
class Preprocessor(nn.Module):
    def __init__(self, lower, upper, mean, std):
        super().__init__()
        self.lower = lower
        self.upper = upper
        self.mean = mean
        self.std = std

    def forward(self, x):
        non_geo = x[:, :6]
        geo = x[:, 6:]
        clamped = torch.clamp(non_geo, min=self.lower, max=self.upper)
        scaled = (clamped - self.mean) / self.std
        return torch.cat([scaled, geo], dim=1)


class Modelo1(nn.Module):
    def __init__(self, prep):
        super().__init__()
        self.prep = prep
        self.fc1 = nn.Linear(8, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.prep(x)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        return self.fc3(x)


class Modelo2(nn.Module):
    def __init__(self, prep):
        super().__init__()
        self.prep = prep
        self.fc1 = nn.Linear(8, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 32)
        self.fc4 = nn.Linear(32, 1)
        self.relu = nn.ReLU()
        self.drop = nn.Dropout(0.1)

    def forward(self, x):
        x = self.prep(x)
        x = self.drop(self.relu(self.fc1(x)))
        x = self.drop(self.relu(self.fc2(x)))
        x = self.drop(self.relu(self.fc3(x)))
        return self.fc4(x)


class Modelo3(nn.Module):
    def __init__(self, prep):
        super().__init__()
        self.prep = prep
        self.fc1 = nn.Linear(8, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, 1)
        self.relu = nn.ReLU()
        self.drop = nn.Dropout(0.2)

    def forward(self, x):
        x = self.prep(x)
        x = self.drop(self.relu(self.fc1(x)))
        x = self.drop(self.relu(self.fc2(x)))
        x = self.drop(self.relu(self.fc3(x)))
        return self.fc4(x)

In [32]:
prep = Preprocessor(lower_bound, upper_bound, train_mean, train_std)

def entrenar(model, x_tr, y_tr, x_v, y_v, epochs, lr):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    for epoch in range(epochs):
        model.train()
        pred = model(x_tr)
        loss = criterion(pred, y_tr)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (epoch + 1) % 50 == 0:
            model.eval()
            with torch.no_grad():
                val_loss = criterion(model(x_v), y_v).item()
            print(f"Epoch {epoch+1}/{epochs} - Train: {loss.item():.4f} - Val: {val_loss:.4f}")

In [33]:
def evaluar(model, x, y):
    model.eval()
    with torch.no_grad():
        preds = model(x).squeeze()
        targets = y.squeeze()
        mae = torch.mean(torch.abs(preds - targets)).item()
        mse = torch.mean((preds - targets) ** 2).item()
        rmse = mse ** 0.5
        ss_res = torch.sum((targets - preds) ** 2)
        ss_tot = torch.sum((targets - targets.mean()) ** 2)
        r2 = (1 - ss_res / ss_tot).item()
    return mae, mse, rmse, r2

In [34]:
torch.manual_seed(42)
modelo1 = Modelo1(prep)
print("Modelo 1: 64-32, lr=0.001, 200 epochs")
entrenar(modelo1, x_train_t, y_train_t, x_val_t, y_val_t, epochs=200, lr=0.001)

Modelo 1: 64-32, lr=0.001, 200 epochs
Epoch 50/200 - Train: 1.1869 - Val: 1.2356
Epoch 100/200 - Train: 1.0271 - Val: 1.0266
Epoch 150/200 - Train: 0.9331 - Val: 0.9374
Epoch 200/200 - Train: 0.8556 - Val: 0.8633


In [35]:
torch.manual_seed(42)
modelo3 = Modelo3(prep)
print("Modelo 3: 256-128-64, lr=0.001, 250 epochs, dropout=0.2")
entrenar(modelo3, x_train_t, y_train_t, x_val_t, y_val_t, epochs=250, lr=0.001)

Modelo 3: 256-128-64, lr=0.001, 250 epochs, dropout=0.2
Epoch 50/250 - Train: 1.5194 - Val: 1.4790
Epoch 100/250 - Train: 1.4003 - Val: 1.3870
Epoch 150/250 - Train: 1.2066 - Val: 1.1104
Epoch 200/250 - Train: 0.9360 - Val: 0.9093
Epoch 250/250 - Train: 0.8049 - Val: 0.7308


In [36]:
torch.manual_seed(42)
modelo2 = Modelo2(prep)
print("Modelo 2: 128-64-32, lr=0.0005, 300 epochs, dropout=0.1")
entrenar(modelo2, x_train_t, y_train_t, x_val_t, y_val_t, epochs=300, lr=0.0005)

Modelo 2: 128-64-32, lr=0.0005, 300 epochs, dropout=0.1
Epoch 50/300 - Train: 1.4775 - Val: 1.3358
Epoch 100/300 - Train: 1.3347 - Val: 1.1798
Epoch 150/300 - Train: 1.1472 - Val: 0.9675
Epoch 200/300 - Train: 0.9234 - Val: 0.7155
Epoch 250/300 - Train: 0.7658 - Val: 0.6058
Epoch 300/300 - Train: 0.7178 - Val: 0.5740


In [37]:
print("validacion:")
print()
for nombre, modelo in [("Modelo 1", modelo1), ("Modelo 2", modelo2), ("Modelo 3", modelo3)]:
    mae, mse, rmse, r2 = evaluar(modelo, x_val_t, y_val_t)
    print(f"{nombre} -> MAE: {mae:.4f}, MSE: {mse:.4f}, RMSE: {rmse:.4f}, R2: {r2:.4f}")

validacion:

Modelo 1 -> MAE: 0.7176, MSE: 0.8633, RMSE: 0.9291, R2: 0.3444
Modelo 2 -> MAE: 0.5555, MSE: 0.5740, RMSE: 0.7576, R2: 0.5641
Modelo 3 -> MAE: 0.6143, MSE: 0.7308, RMSE: 0.8549, R2: 0.4450


In [38]:
mejor = modelo2

mae, mse, rmse, r2 = evaluar(mejor, x_train_t, y_train_t)
print(f"train -> MAE: {mae:.4f}, MSE: {mse:.4f}, RMSE: {rmse:.4f}, R2: {r2:.4f}")

mae, mse, rmse, r2 = evaluar(mejor, x_val_t, y_val_t)
print(f"val   -> MAE: {mae:.4f}, MSE: {mse:.4f}, RMSE: {rmse:.4f}, R2: {r2:.4f}")

mae, mse, rmse, r2 = evaluar(mejor, x_test_t, y_test_t)
print(f"test  -> MAE: {mae:.4f}, MSE: {mse:.4f}, RMSE: {rmse:.4f}, R2: {r2:.4f}")

train -> MAE: 0.5465, MSE: 0.5583, RMSE: 0.7472, R2: 0.5824
val   -> MAE: 0.5555, MSE: 0.5740, RMSE: 0.7576, R2: 0.5641
test  -> MAE: 0.5467, MSE: 0.5674, RMSE: 0.7533, R2: 0.5647
